<a href="https://colab.research.google.com/github/BatHands/AI/blob/main/RAG_with_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
%pip install -qq docling python-dotenv langchain langchain-milvus langchain-text-splitters langchain-huggingface pymilvus[milvus_lite]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 6.4 MB/s eta 0:00:00


In [3]:
import os

from dotenv import load_dotenv

load_dotenv()

False

In [4]:
from typing import Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument

from docling.document_converter import DocumentConverter

class DoclingPDFLoader(BaseLoader):

    def __init__(self, file_path: str | list[str]) -> None:
      self._file_paths = file_path if isinstance(file_path, list) else [file_path]
      self._converter = DocumentConverter()

    def lazy_load(self) -> Iterator[LCDocument]:
      for source in self._file_paths:
        dl_doc = self._converter.convert(source).document
        text = dl_doc.export_to_markdown()
        yield LCDocument(page_content=text)

In [12]:
FILE_PATH = "https://ontheline.trincoll.edu/images/bookdown/sample-local-pdf.pdf"

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DoclingPDFLoader(file_path=FILE_PATH)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

In [14]:
docs = loader.load()
splits = text_splitter.split_documents(docs)

In [16]:
!pip install huggingface_hub
!pip install ipywidgets
from huggingface_hub import notebook_login
notebook_login()

In [18]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

HF_EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
embeddings = HuggingFaceEmbeddings(model_name=HF_EMBED_MODEL_ID)

In [23]:
from tempfile import TemporaryDirectory

from langchain_milvus import Milvus

MILVUS_URI = os.environ.get(
    "MILVUS_URI", f"{(tmp_dir := TemporaryDirectory()).name}/milvus_demo.db"
)

vectorstore = Milvus.from_documents(
    splits,
    embeddings,
    connection_args={"uri": MILVUS_URI},
    drop_old=True,
)

In [51]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

HF_API_KEY = os.environ.get("HF_API_KEY")
HF_LLM_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

llm = HuggingFaceEndpoint(
    repo_id=HF_LLM_MODEL_ID,
    huggingfacehub_api_token=HF_API_KEY,
    task="text-generation",
)

chat_model = ChatHuggingFace(llm=llm)

In [57]:
from typing import Iterable

from langchain_core.documents import Document as LCDocument
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs: Iterable[LCDocument]):
  return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
  SystemMessagePromptTemplate.from_template("You're a helpful assistant."),
  HumanMessagePromptTemplate.from_template("Context information is below.\n---------------------\n{context}\n---------------------\nGiven the context information and not prior knowledge, answer the query.\nQuery: {question}\nAnswer:\n"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

In [59]:
rag_chain.invoke("Summarize the sample PDF in 2 sentences")

' The sample PDF is a three-page document written in Latin, containing text about various design elements and colors. It also includes a few sentences in English, such as "Lorem ipsum dolor sit amet," which is a placeholder text commonly used in the publishing and typesetting industry.'